In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
from pandas import ExcelWriter
import datetime
from time import sleep
import os



In [3]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'SM BCSM' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.2")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

# scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

# %%


Running SM BCSM Web Scraping Tool v.1.2


In [4]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()


In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={
                
                'SM BCSM 1':'https://www.bcsm.sm/site/home/funzioni/registri-e-albi/registro-dei-soggetti-autorizzati/imprese-finanziarie-sammarinesi.html' , 
         
		'SM BCSM 3': 'https://www.bcsm.sm/site/en/home/functions/registers/register-of-insurance-and-reinsurance-brokers.html', 
   
		'SM BCSM 5': 'https://www.bcsm.sm/site/en/home/functions/registers/register-of-financial-promoters.html', 
   
		'SM BCSM 6': 'https://www.bcsm.sm/site/en/home/functions/registers/register-of-trustees.html'
        }

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')



In [6]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def click_on_cookies(web_driver):

    try:

        web_driver.find_element(By.XPATH,f'//*[@id="c-bns"]').click()
        print('[Success] : Success to Click Cookie')

    except Exception as err:

        print('[ERROR] : Failed to click "I Accept" button on the cookies banner:', err)


def scroll_to_bottom(driver):

    # Get scroll height

    last_height = driver.execute_script("return document.body.scrollHeight")



    while True:


        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")



        sleep(1)



        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:

            break

        last_height = new_height
        

In [7]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------
for k, reg in enumerate(regdict):

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    
    driver.get(regdict[reg])
    
    if reg =='SM BCSM 1':
        click_on_cookies(driver)
        
        clinks = []
    
        sleep(4)

        soup=BeautifulSoup(driver.page_source, 'html.parser')
        
        lista = soup.find("div", {"class":"lista"})

        rows=soup.find_all("div",{"class":"row list smallsize"})
        
        for row in rows:
            
            a = row.find("a")
            
            clinks.append('https://www.bcsm.sm' + a['href'] if a['href'][0] != "/" else "https://www.bcsm.sm" + a['href'])

            sqldict['Name'].append(a.text.strip())
            
            sqldict['ListValidityDate'].append(row.find_all("div")[1].text)
            
            sqldict['InternalID_1'].append(row.find_all("div")[2].text)
            
            sqldict['InternalID_1_type'].append('N. iscrizione (Registro Soggetti Autorizzati)')
            
            sqldict['RegulationType'].append('Registered')
            
            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])
            
            sqldict['ListProcessDate'].append(processdate)
            
        for link in clinks:
            
            driver.get(link)
            
            soup_inner=BeautifulSoup(driver.page_source, 'html.parser')
            
            table = soup_inner.find("div", {"class":"container-fluid"})
            
            sleep(3)
            
            try:
                rows_inner = table.find_all('div',{"class":"row"})
            except:
                scroll_to_bottom(driver)
                sleep(5)
                rows_inner = table.find_all('div',{"class":"row"})
            
            info = rows_inner[3]
            
            sqldict['InternalID_2_type'].append('Codice Operatore Economico')
            
            sqldict['InternalID_2'].append(info.find_all('div',{"class":"col-md-7 valore"})[2].text)
    
    elif reg =='SM BCSM 3':
        
        soup=BeautifulSoup(driver.page_source, 'html.parser')
        
        content_list = []
        
        for tr in soup.select('tbody tr'):
            # Find all <td> elements within the current <tr>
            contents = tr.find_all('td')
        
            for content in contents:
                # Remove the <strong> tags and print the text content
                for strong in content.find_all('strong'):
                    
                    strong.decompose()
                
                if not (content.text.strip()==''):
                
                    content_list.append(content.text.strip())
                
                    
        for i in range(0,len(content_list)-4,4):
           
            name = content_list[i]
           
            occupation = content_list[i+1]
           
            registration_numbver = content_list[i+2]
           
            state = content_list[i+3]
            
            if state == 'ACTIVE':
                sqldict['RegulationType'].append('Registered')
            elif state == 'INACTIVE':
                sqldict['RegulationType'].append('Inactive')
            
            sqldict['Name'].append(name)
            
            sqldict['InternalID_1'].append(registration_numbver)
            
            sqldict['InternalID_1_type'].append('N. iscrizione (Registro Soggetti Autorizzati)')
            
            #sqldict['Typology'].append(occupation)
            
            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])
            
            sqldict['ListProcessDate'].append(processdate)
            
    elif reg == 'SM BCSM 5':
        
        soup=BeautifulSoup(driver.page_source, 'html.parser')
        
        content_list = []
        
        for tr in soup.select('tbody tr'):
            # Find all <td> elements within the current <tr>
            contents = tr.find_all('td')
        
            for content in contents:
                # Remove the <strong> tags and print the text content
                for strong in content.find_all('strong'):
                    
                    strong.decompose()
                
                if not (content.text.strip()==''):
                
                    content_list.append(content.text.strip())
                    
        # Search for the target string
        target_string = "List of CANCELLED financial promoters"

        # Use the .index() method to find the index of the target string
        index = content_list.index(target_string)
        
        content_list.pop(index)        
                    
        for i in range(0,len(content_list)-2,3):
           
            name = content_list[i]
            
            registration_numbver = content_list[i+1]
            
            state = content_list[i+2]
            
            if state == 'ACTIVE':
                sqldict['RegulationType'].append('Registered')
            elif state == 'INACTIVE':
                sqldict['RegulationType'].append('Inactive')
            
            sqldict['Name'].append(name)
            
            sqldict['InternalID_1'].append(registration_numbver)
            
            sqldict['InternalID_1_type'].append('N. iscrizione (Registro Soggetti Autorizzati)')
            
            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])
            
            sqldict['ListProcessDate'].append(processdate)
            
        inner_links = []
        for a in soup.select('tbody tr a'):
            if a['href'].split('/')[-1].startswith('a'):
                print(a['href'])
                inner_links.append('https://www.bcsm.sm' + a['href'])          
        for inner in inner_links:
            inner_content = []
            driver.get(inner)
            soup=BeautifulSoup(driver.page_source, 'html.parser')
            for tr in soup.select('tbody tr'):
            # Find all <td> elements within the current <tr>
                contents = tr.find_all('td')
                for content in contents:
                    # Remove the <strong> tags and print the text content
                    for strong in content.find_all('strong'):
                        
                        strong.decompose()
                    
                    if not (content.text.strip()==''):
                    
                        inner_content.append(content.text.strip())
                        
                for i in range(0,len(inner_content)-2,3):
                
                    name = inner_content[i]
                    
                    registration_numbver = inner_content[i+1]
                    
                    state = inner_content[i+2]
                    
                    # sqldict['Name'].append(name)
                    
                    # sqldict['InternalID_1'].append(str(registration_numbver))
                    
                    # print(registration_numbver)
                    
                    # sqldict['InternalID_1_type'].append('Registration Number')
                    
                    # sqldict['RegulationType'].append(state)
                    
                    # sqldict['RegCtry'].append(reg.split(' ')[0]) 

                    # sqldict['RegCode'].append(reg.split(' ')[1])

                    # sqldict['ListCode'].append(reg.split(' ')[-1])
                    
                    # sqldict['ListProcessDate'].append(processdate)
                
    elif reg == 'SM BCSM 6':
        
        soup=BeautifulSoup(driver.page_source, 'html.parser')
        
        content_list = []
        
        for tr in soup.select('tbody tr'):
            # Find all <td> elements within the current <tr>
            contents = tr.find_all('td')
            for content in contents:    

                content_list.append(content.text.strip())
                
        for i in range(0,len(content_list)-8,9):
           
            registration_numbver = content_list[i]
            
            name = content_list[i+1]
           
            Headquarters = content_list[i+2].replace('\t','')
           
            eoc_number = content_list[i+4]
            
            Authorisation_date = content_list[i+5]
           
            state = content_list[i+7]
            
            if state == 'ACTIVE':
                sqldict['RegulationType'].append('Registered')
            elif state == 'INACTIVE':
                sqldict['RegulationType'].append('Inactive')
            
            sqldict['Name'].append(name)
            
            sqldict['InternalID_1'].append(registration_numbver)
            
            sqldict['InternalID_1_type'].append('N. iscrizione (Registro Soggetti Autorizzati)')
            
            sqldict['InternalID_2'].append(eoc_number)
            
            sqldict['InternalID_2_type'].append('Codice Operatore Economico')
            
            sqldict['ListValidityDate'].append(Authorisation_date)
            
            # sqldict['RegulationType'].append(state)
            
            sqldict['Address_1'].append(Headquarters)        
            
            sqldict['RegCtry'].append(reg.split(' ')[0]) 
            
            sqldict['Cntry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1])
            
            sqldict['ListProcessDate'].append(processdate)
            
        
          
    sqldict = bourange_same_length_array(sqldict)
                
            

            


[INFO] : Working 1/4 _(SM BCSM 1)_ 
[ERROR] : Failed to click "I Accept" button on the cookies banner: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//*[@id="c-bns"]"}
  (Session info: chrome=147.0.7727.138); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff741d3a8e5+14e45]
	chromedriver!GetHandleVerifier [0x7ff741d3a950+14eb0]
	chromedriver!(No symbol) [0x7ff741aad6ed]
	chromedriver!(No symbol) [0x7ff741b06cfe]
	chromedriver!(No symbol) [0x7ff741b0700c]
	chromedriver!(No symbol) [0x7ff741b57cb7]
	chromedriver!(No symbol) [0x7ff741b5483b]
	chromedriver!(No symbol) [0x7ff741af90e8]
	chromedriver!(No symbol) [0x7ff741af9fc3]
	chromedriver!GetHandleVerifier [0x7ff742050149+32a6a9]
	chromedriver!GetHandleVerifier [0x7ff74204a715+324c75]
	chromedriver!GetHandleVerifier [0x7ff74206c012+346572]
	chromedriver!GetHandl

ValueError: 'List of CANCELLED financial promoters' is not in list

In [ ]:
os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\10\ipykernel_36732\512081806.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
df.to_csv('total_new4.csv')

In [ ]:
for key, value in sqldict.items():
    print(f"Length of array under key '{key}': {len(value)}")

Length of array under key 'bvdid': 12
Length of array under key 'priority': 12
Length of array under key 'ListLabel': 12
Length of array under key 'Typology': 12
Length of array under key 'EntryType': 12
Length of array under key 'Name': 12
Length of array under key 'InternalID_1': 12
Length of array under key 'InternalID_1_type': 12
Length of array under key 'InternalID_2': 12
Length of array under key 'InternalID_2_type': 12
Length of array under key 'InternalID_3': 12
Length of array under key 'InternalID_3_type': 12
Length of array under key 'CoType': 12
Length of array under key 'License_Type': 12
Length of array under key 'Address_1': 12
Length of array under key 'Address_2': 12
Length of array under key 'City': 12
Length of array under key 'Zip': 12
Length of array under key 'Cntry': 12
Length of array under key 'Phone': 12
Length of array under key 'Fax': 12
Length of array under key 'Website': 12
Length of array under key 'Email': 12
Length of array under key 'RegulationType':